In [11]:

%%writefile auditor.py

import os
import csv
from datetime import datetime, timedelta

def scan_folder(folder_path):

    files = []

    for folder, subfolders, filenames in os.walk(folder_path):
        for name in filenames:

            file_path = os.path.join(folder, name)

            size = os.path.getsize(file_path)
            modified_time = os.path.getmtime(file_path)

            modified_date = datetime.fromtimestamp(modified_time)
            days_old = (datetime.now() - modified_date).days

            if days_old > 30:
                status = "Old File"
            else:
                status = "OK"

            file_info = {
                "File Name": name,
                "Size (Bytes)": size,
                "File Type": os.path.splitext(name)[1],
                "Last Modified": modified_date.strftime("%Y-%m-%d"),
                "Days Old": days_old,
                "Status": status
            }

            files.append(file_info)

    return files


Overwriting auditor.py


In [12]:

%%writefile audit.py

import csv
from datetime import datetime
from auditor import scan_folder

folder_path = "project_folder"

fine_per_day = 10

files = scan_folder(folder_path)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

audit_report = "audit_report_" + timestamp + ".csv"
penalties_report = "penalties_report_" + timestamp + ".csv"

# Audit report
with open(audit_report, "w", newline="") as file:
    writer = csv.DictWriter(file, fieldnames=[
        "File Name", "Size (Bytes)", "File Type",
        "Last Modified", "Days Old", "Status"
    ])

    writer.writeheader()
    writer.writerows(files)

# Penalties report
with open(penalties_report, "w", newline="") as file:
    writer = csv.writer(file)

    writer.writerow(["File Name", "Days Old", "Fine (Rs.)"])

    for item in files:
        if item["Days Old"] > 30:
            extra_days = item["Days Old"] - 30
            fine = extra_days * fine_per_day

            writer.writerow([
                item["File Name"],
                item["Days Old"],
                fine
            ])

print("Audit completed successfully!")
print("Total files:", len(files))
print("Audit report:", audit_report)
print("Penalties report:", penalties_report)


Overwriting audit.py


In [13]:

import os
import time

os.makedirs("project_folder", exist_ok=True)

with open("project_folder/main.py", "w") as file:
    file.write("print('Hello World')")

with open("project_folder/data.txt", "w") as file:
    file.write("Sample project data")

# Make one file 40 days old
old_file = "project_folder/data.txt"

old_time = time.time() - (40 * 24 * 60 * 60)

os.utime(old_file, (old_time, old_time))

print("Sample project folder is ready!")


Sample project folder is ready!


In [14]:

%run audit.py


Audit completed successfully!
Total files: 2
Audit report: audit_report_20261020_165957.csv
Penalties report: penalties_report_20261020_165957.csv


In [16]:

import pandas as pd
import glob

audit_file = sorted(glob.glob("audit_report_*.csv"))[-1]
penalty_file = sorted(glob.glob("penalties_report_*.csv"))[-1]

print("AUDIT REPORT")
display(pd.read_csv(audit_file))

print("PENALTIES REPORT")
display(pd.read_csv(penalty_file))


AUDIT REPORT


,File Name,Size (Bytes),File Type,Last Modified,Days Old,Status
0,data.txt,19,.txt,2026-09-10,40,Old File
1,main.py,20,.py,2026-10-20,0,OK


PENALTIES REPORT


,File Name,Days Old,Fine (Rs.)
0,data.txt,40,100
